# Handoffs and multi-agent design

## Learning goals

- Understand handoffs as transfer of active-agent ownership
- Give specialists clear descriptions and non-overlapping responsibilities
- Attach typed metadata and callbacks to a handoff
- Compare handoffs with `Agent.as_tool()` manager patterns
- Inspect the final agent and reason about guardrail/history boundaries


## What a handoff changes

A handoff is exposed to the model as a transfer tool. When selected, the SDK changes the active agent and continues the same run with the receiving specialist. The specialist can then use its own instructions, tools, model settings, handoffs, and output type.

Use a handoff when the specialist should own the remaining conversation. Use an agent as a tool when a manager should call a specialist for a bounded subtask, receive its result, and retain control for synthesis.


## Mental model

```mermaid
sequenceDiagram
  participant U as User
  participant T as Triage agent
  participant R as Runner
  participant S as Specialist agent
  U->>T: Travel request
  T-->>R: transfer_to_specialist(...)
  R->>R: Validate handoff and optional metadata
  R->>S: Continue run with transferred history
  S-->>U: Specialist final output
```

The triage agent does not automatically return after a handoff. `result.last_agent` identifies which agent finished and is often the best starting agent for the next user turn.


## Define specialists with clear boundaries

`handoff_description` helps the routing agent decide when a specialist is appropriate. Descriptions that all say “helps with travel” create ambiguity; name the specific responsibility and exclusions.


In [ ]:
import os

from agents import Agent, ModelSettings

MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
SETTINGS = ModelSettings(max_tokens=350, store=False)

budget_agent = Agent(
    name="Budget specialist",
    handoff_description="Estimates transparent travel costs and states pricing assumptions.",
    instructions=(
        "Estimate ranges, list assumptions, and never claim live prices unless a tool confirms them."
    ),
    model=MODEL,
    model_settings=SETTINGS,
)

itinerary_agent = Agent(
    name="Itinerary specialist",
    handoff_description="Builds paced day-by-day itineraries after destination needs are clear.",
    instructions="Create practical itineraries with travel time and rest breaks.",
    model=MODEL,
    model_settings=SETTINGS,
)

safety_agent = Agent(
    name="Travel safety specialist",
    handoff_description="Handles packing, accessibility, and general travel-safety questions.",
    instructions=(
        "Give general, cautious guidance and recommend authoritative local sources for current conditions."
    ),
    model=MODEL,
    model_settings=SETTINGS,
)

print([agent.name for agent in [budget_agent, itinerary_agent, safety_agent]])


## Register outgoing handoffs on triage

Passing an `Agent` directly creates a default handoff tool. The triage instructions should explain when it may answer directly versus transfer. Avoid chains of specialists that can hand off in circles unless the runtime has deliberate cycle controls.


In [ ]:
triage_agent = Agent(
    name="Travel triage",
    instructions=(
        "Identify the primary need. Transfer detailed cost questions to Budget specialist, "
        "day-by-day planning to Itinerary specialist, and packing/accessibility/safety "
        "questions to Travel safety specialist. Ask one clarification when the destination "
        "or request type is genuinely ambiguous."
    ),
    model=MODEL,
    model_settings=SETTINGS,
    handoffs=[budget_agent, itinerary_agent, safety_agent],
)

print("Triage handoff count:", len(triage_agent.handoffs))
print([handoff_agent.name for handoff_agent in triage_agent.handoffs])


## Inspect the handoff tool contract

The `handoff()` helper gives access to the generated transfer tool name, description, input schema, filters, enablement, and callback. Plain handoffs have an empty argument object by default.


In [ ]:
from agents import handoff

itinerary_handoff = handoff(itinerary_agent)
print("Tool name:", itinerary_handoff.tool_name)
print("Description:", itinerary_handoff.tool_description)
print("Input schema:", itinerary_handoff.input_json_schema)


## Add typed handoff metadata

`input_type` describes small model-generated metadata for the transfer—reason, priority, language, or summary. It does not replace the specialist's conversation input, and it is not trusted application state. Existing user identity or database handles belong in `RunContextWrapper.context`.


In [ ]:
from dataclasses import dataclass, field
from typing import Literal

from pydantic import BaseModel, Field
from agents import RunContextWrapper


class HandoffMetadata(BaseModel):
    reason: str = Field(min_length=5, max_length=160)
    priority: Literal["normal", "high"] = "normal"


@dataclass
class AppContext:
    audit_events: list[dict[str, str]] = field(default_factory=list)


async def record_budget_handoff(
    ctx: RunContextWrapper[AppContext],
    input_data: HandoffMetadata,
) -> None:
    ctx.context.audit_events.append({
        "event": "handoff",
        "to": budget_agent.name,
        "priority": input_data.priority,
    })


budget_handoff = handoff(
    agent=budget_agent,
    on_handoff=record_budget_handoff,
    input_type=HandoffMetadata,
    tool_name_override="transfer_to_budget_specialist",
    tool_description_override="Transfer detailed cost-estimation work to the budget specialist.",
)

print(budget_handoff.tool_name)
print(budget_handoff.input_json_schema)


## Test the handoff callback offline

The low-level invocation validates the JSON metadata, runs the callback, and returns the fixed destination agent. It does not call a model.


In [ ]:
app_context = AppContext()
destination = await budget_handoff.on_invoke_handoff(
    RunContextWrapper(context=app_context),
    '{"reason": "The user requested a detailed daily budget.", "priority": "normal"}',
)

print("Destination:", destination.name)
print("Audit events:", app_context.audit_events)
assert destination is budget_agent


## Handoff or agent-as-tool?

| Requirement | Handoff | `Agent.as_tool()` |
|---|---|---|
| Specialist owns remaining turn | Yes | No |
| Manager synthesizes several specialists | Awkward | Yes |
| `result.last_agent` becomes specialist | Yes | Manager normally remains active |
| Specialist exposed as transfer tool | Yes | Exposed as function-like tool |
| Best for | Routing/escalation | Manager-worker/orchestrator pattern |


In [ ]:
budget_tool = budget_agent.as_tool(
    tool_name="estimate_travel_budget",
    tool_description="Ask the budget specialist for one bounded cost estimate.",
    max_turns=4,
)

manager_agent = Agent(
    name="Travel manager",
    instructions="Coordinate specialist input and produce one final answer.",
    model=MODEL,
    model_settings=SETTINGS,
    tools=[budget_tool],
)
print([tool.name for tool in manager_agent.tools])


In [ ]:
from agents import Runner

RUN_LIVE_CALL = False

if RUN_LIVE_CALL:
    result = await Runner.run(
        triage_agent,
        "Estimate a realistic two-day Jaipur budget and list assumptions.",
        max_turns=6,
    )
    print(result.final_output)
    print("Answered by:", result.last_agent.name)
else:
    print("Live handoff run skipped.")


## Important boundaries

- Input guardrails attached to agents apply to the first agent in the chain; output guardrails apply to the agent producing the final output.
- Function-tool guardrails do not wrap the handoff pipeline itself.
- The receiving agent sees conversation history unless an input filter or history configuration changes it. Minimize unnecessary sensitive context.
- Handoffs remain within one run and count toward runner turns.
- Final output type may change with the final specialist; normalize or validate the possible contracts.
- A handoff callback is a good audit hook, not a substitute for authorization.


## Failure cases and improvements

- **Overlapping specialists:** tighten descriptions, examples, and route evaluation cases.
- **Handoff ping-pong:** restrict outgoing handoffs and enforce finite `max_turns`.
- **Triage over-answers:** explicitly define when it must transfer.
- **Specialist receives excessive history:** use an input filter or nested handoff history deliberately.
- **Typed output surprise:** inspect `last_agent` before assuming a final output type.
- **Sensitive handoff logs:** store allowlisted metadata rather than full conversation text.


## Exercise

1. Add a language specialist with a non-overlapping handoff description.
2. Write three routing cases for each specialist and one ambiguous case.
3. Add `is_enabled` to disable budget transfer for an unauthorized context.
4. Compare a manager with two `as_tool()` specialists against triage handoffs.
5. Decide what history the safety specialist actually needs and design an input filter.


## Official references

- [Agents SDK handoffs](https://openai.github.io/openai-agents-python/handoffs/)
- [Multi-agent patterns](https://openai.github.io/openai-agents-python/multi_agent/)
- [Agent configuration](https://openai.github.io/openai-agents-python/agents/)


## Summary

Handoffs transfer active ownership to a specialist; agents-as-tools let a manager retain control. Define narrow specialist roles, provide useful handoff descriptions, validate optional transfer metadata, audit the boundary safely, cap turns, and inspect `result.last_agent` before interpreting final output.
